# Médias móveis como régua — Etapa 2 do plano de previsão

**Objetivo:** medir quanto regras simples acertam o **total vendido nos 3 meses seguintes** de cada produto × loja, onde erram e qual o viés delas. A melhor régua vira a nota mínima que o modelo precisa superar, e os erros mostram quais features valem a pena montar na Etapa 3.

**Resposta curta:** a previsão de cada produto × loja para os próximos 3 meses é **média da categoria no local × índice sazonal × tendência**. Ela vem com dois **limites de venda**: até quanto se vende em 3 de cada 4 vezes e em 9 de cada 10 vezes. Nos 12 meses de teste, a venda real ficou dentro desses limites em 75% e 90% dos casos, como prometido. Somando os produtos, a previsão erra **10,7% por categoria × local** e **6,1% por local**, com viés de −2,8%. A seção 14 é a **base de comparação** para qualquer previsão nova. O achado central: **o histórico de um produto não informa quanto ele vai vender**, e o que prevê é o nível do grupo, a estação e a tendência.

**De onde vem o desenho** (notebooks 01 e 02):

| O que já sabemos | Como entra aqui |
|---|---|
| Série mensal, alvo = total dos 3 meses seguintes | Cada linha é uma série × mês de corte, e o alvo soma os 3 meses depois do corte |
| Nenhuma série nova, e 98% das "paradas" voltam a vender | Todas as 6.054 séries entram, sem regras separadas |
| Mesma estação em todas as categorias | Um índice sazonal geral |
| +15% de demanda no último ano | Medir o viés das médias e testar uma correção de tendência |
| Séries muito parecidas entre si; 93% do volume vem de produtos vendidos nos 6 locais | Testar se o nível vem da própria série, do produto nos 6 locais ou da categoria no local |

**Termos:**

| Termo | O que quer dizer, em palavras simples |
|---|---|
| Corte | O "hoje" do teste: o mês em que fingimos estar fazendo a previsão. Só pode usar o que já tinha acontecido até esse mês, nunca o futuro |
| Alvo | Quanto realmente foi vendido nos 3 meses seguintes ao corte — o número que a previsão está tentando acertar |
| MM12 / MM3 | A venda média por mês, olhando para os últimos 12 meses (MM12) ou só os últimos 3 (MM3) |
| Índice sazonal | Um número que diz se um mês vende mais ou menos que a média do ano. 1,5 = esse mês vende 50% a mais que a média; 0,5 = vende metade da média |
| Fator de tendência | Um número um pouco maior que 1 que "empurra" a previsão para cima, para compensar o fato de a demanda estar crescendo com o tempo |
| Erro (WAPE) | De cada 100 unidades vendidas, quantas a previsão errou para mais ou para menos. Erro de 50% = errou 50 unidades a cada 100 vendidas. Prever sempre "zero" já dá 100% de erro |
| Viés | Diz se a previsão costuma ficar para cima ou para baixo da venda real, em média. Viés de −10% = a previsão, no total, fica 10% abaixo do que de fato foi vendido |
| Limite de venda (`q_75`, `q_90`) | Um teto: "a venda não deve passar disso". `q_75` = teto que segura a venda real em 3 de cada 4 vezes; `q_90` = teto que segura em 9 de cada 10 vezes |
| Desvio de Poisson | Outra forma de medir o erro por produto, parecida com o erro (WAPE), mas que não dá vantagem para quem "chuta baixo" (explicado na seção 10) |

Os cálculos ficam em `src/analytics_ai/demand_baselines.py`.

## Resumo em palavras simples (sem estatística)

Imagine que você quer saber quantos motores de popa uma loja vai vender nos próximos 3 meses. Dá pra pensar assim, passo a passo:

1. **Não olhe só para aquele motor específico.** Um produto sozinho vende pouco e de forma irregular — um mês vende 3, no outro vende 0, no outro vende 8. Esse vaivém é sorte, não é um padrão que se repita (isso é o que a seção 8 mostra: quem vendeu mais no ano passado não é quem vai vender mais este ano). Então, em vez de usar o histórico *daquele* motor, usamos a média de venda de **todos os motores daquela loja**. Um grupo maior de produtos "balança" muito menos que um produto isolado, então essa média é uma base bem mais confiável.
2. **Ajuste pela época do ano.** Se estamos prevendo dezembro a fevereiro (verão, alta temporada de barcos), a previsão precisa ser maior do que se estivéssemos prevendo junho a agosto. Esse ajuste é o "índice sazonal".
3. **Ajuste pelo crescimento.** As vendas da loja estão crescendo cerca de 15% ao ano. Se você só olhar a média do que já vendeu, vai sempre chutar um pouco abaixo do que realmente vai vender, porque o passado é sempre "menor" que o presente numa empresa que está crescendo. Por isso multiplicamos por um fatorzinho de tendência para compensar.

Juntando os três passos (média do grupo × ajuste de época × ajuste de crescimento) chegamos num número esperado de venda.

**Só o número esperado não basta**, porque a venda de um produto varia muito ao redor dele. Por isso a previsão também entrega dois "tetos":
- um teto que a venda real ultrapassa só 1 em cada 4 vezes;
- um teto mais alto que a venda real ultrapassa só 1 em cada 10 vezes.

Pense nisso como uma previsão do tempo: "a temperatura deve ficar perto de 25°C, mas pode chegar a 30°C algumas vezes, e a 35°C bem raramente". Testamos esses tetos nos 12 últimos meses de dados reais, e eles se confirmaram: a venda ficou dentro do primeiro teto em 75% dos casos e dentro do segundo em 90% — exatamente como o esperado.

**Sobre o erro:** para um produto isolado numa loja, a previsão erra bastante (a venda de um item é quase um jogo de sorte). Mas quando juntamos vários produtos — por exemplo, todos os motores de uma loja, ou tudo o que uma loja vende — os erros de cada produto se cancelam entre si, e o erro total fica pequeno: 10,7% quando somamos por categoria numa loja, e só 6,1% quando somamos tudo o que uma loja vende. Isso é ótimo para quem decide compra de estoque por categoria ou planeja por loja.

**A descoberta mais importante do notebook:** olhar para o histórico de um produto específico não ajuda a prever o futuro dele. Um produto que vendeu muito mais que outro no ano passado não tem mais chance de continuar vendendo mais — é como jogar uma moeda: quem tirou cara 3 vezes seguidas não tem mais chance de tirar cara na próxima. O que realmente prevê a venda é o grupo (categoria + loja), a época do ano e a tendência de crescimento — não o histórico individual do produto.

A seção 14, no fim do notebook, junta esses números numa "prova" que qualquer modelo de machine learning mais sofisticado, feito depois, vai precisar superar para valer a pena ser usado.

## Setup

In [ ]:
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Sobe a partir do diretório do notebook até achar a raiz do bundle (databricks.yml)."""
    for candidate in (start, *start.parents):
        if (candidate / "databricks.yml").exists():
            return candidate
    raise FileNotFoundError(f"databricks.yml não encontrado acima de {start}")


# Funciona no workspace (.bundle/.../files/notebooks/ml) e localmente.
PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import pandas as pd

from src.analytics_ai.demand_baselines import (
    add_forecasts,
    add_months,
    aggregated_error_table,
    apply_interval_buckets,
    build_cutoff_panel,
    comparison_scorecard,
    cutoff_months,
    error_table,
    fit_interval_buckets,
    interval_coverage,
    monthly_totals,
    past_volume_quintiles,
    seasonal_index_at,
    trend_factor_at,
    year_over_year_persistence,
)
from src.analytics_ai.eda_demand_distribution import add_frequency_class, complete_months, reference_month
from src.analytics_ai.ml_config import MlFeatureConfig

# No Databricks o notebook já expõe `spark`; localmente cria via Databricks Connect
# (defina DATABRICKS_PROFILE com o nome do seu profile em ~/.databrickscfg).
spark = globals().get("spark")
if spark is None:
    from src.ingestion.utils import build_spark_session

    spark = build_spark_session()

# Databricks e Jupyter injetam `display`; fora deles, cai para print.
display = globals().get("display", print)

config = MlFeatureConfig.from_environment_monthly()
DATE_COL = config.date_col

complete = complete_months(spark.table(config.source_table), DATE_COL)
products = spark.table(config.gold_table("dim_products")).select("sk_product", "sku", "product_name", "category_name")
locations = spark.table(config.gold_table("dim_locations")).select("sk_location", "location_name")

LAST_MONTH = reference_month(complete, DATE_COL)
# Teste: os 12 últimos cortes cujo alvo de 3 meses já aconteceu.
CUTOFFS = cutoff_months(LAST_MONTH, n_cutoffs=12)
# Calibração da faixa (seção 11): 24 cortes cujo alvo termina antes do primeiro corte de teste.
CALIBRATION_CUTOFFS = cutoff_months(CUTOFFS[0], n_cutoffs=24)

print(f"último mês completo: {LAST_MONTH}")
print(f"cortes de teste: {CUTOFFS[0]} a {CUTOFFS[-1]} ({len(CUTOFFS)} cortes)")
print(f"cortes de calibração da faixa: {CALIBRATION_CUTOFFS[0]} a {CALIBRATION_CUTOFFS[-1]} ({len(CALIBRATION_CUTOFFS)} cortes)")

## 1. O painel de teste

Uma linha por série × mês de corte (6.054 séries × 12 cortes), com as somas dos últimos 12 e 3 meses, os meses com venda no último ano e o alvo. A frequência de venda (alta, média, baixa) é calculada **no corte**, com as mesmas faixas do notebook 02.

**Conferência:** `janelas_ok` confirma que cada janela cobre exatamente os 12 meses anteriores e os 3 seguintes, sem buracos. Tem que ser 1,0 em todos os cortes.

In [ ]:
panel_all = (
    add_frequency_class(build_cutoff_panel(complete, CALIBRATION_CUTOFFS + CUTOFFS, DATE_COL))
    .join(products, "sk_product")
    .join(locations, "sk_location")
    .toPandas()
)
totals = monthly_totals(complete, DATE_COL)

# As seções 1 a 10 usam só os cortes de teste; os de calibração entram na seção 11.
panel = panel_all[panel_all["cutoff"].isin(CUTOFFS)]

panel.groupby("cutoff").agg(
    series=("sk_product", "size"),
    janelas_ok=("window_ok", "mean"),
    vendido_nos_3_meses=("target_3m", "sum"),
    alvos_sem_venda=("target_3m", lambda s: round((s == 0).mean(), 3)),
)

**Resultado** *(16/09/2026)*

- 12 cortes (junho/2025 a maio/2026) × 6.054 séries = **72.648 linhas**, com `janelas_ok` = 1,0 em todos os cortes.
- O alvo muda muito com a época: cortes de outubro a dezembro preveem a alta temporada (~51 mil unidades, 21% dos alvos sem venda); cortes de abril e maio preveem a baixa (17 a 20 mil unidades, 55–60% sem venda).

**O que significa:** o painel está correto, e a época prevista pesa muito no tamanho do alvo. O índice sazonal é indispensável.

## 2. Índice sazonal e tendência em cada corte

Recalculados com os meses até cada corte, como aconteceria na prática. A primeira tabela mostra o índice de cada mês do ano; a segunda, o crescimento dos 12 meses até o corte contra os 12 anteriores e o **fator de tendência**.

**Fator de tendência:** a MM12 olha para um período centrado 5,5 meses antes do corte, e os 3 meses previstos ficam centrados 2 meses depois, uma distância de 7,5 meses. O fator é quanto a demanda cresce nesse intervalo, no ritmo de crescimento do corte (crescimento ^ 7,5/12).

In [ ]:
index_by_cutoff = pd.DataFrame({c: seasonal_index_at(totals, c) for c in CUTOFFS}).T.round(2)
index_by_cutoff.index.name = "corte"
display(index_by_cutoff)

trend_by_cutoff = pd.DataFrame(
    [{"corte": c, "crescimento_12m": g, "fator_tendencia": f} for c in CUTOFFS for g, f in [trend_factor_at(totals, c)]]
).round(3)
trend_by_cutoff

**Resultado**

- O índice sazonal quase não muda entre cortes: janeiro fica entre 1,52 e 1,67 e julho entre 0,39 e 0,43. Outubro fica entre 1,02 e 1,11 com o histórico até cada corte, e em 0,99 com o histórico completo: é o mês de fronteira.
- O crescimento em 12 meses vai de **+13% a +19%** conforme o corte, e o fator de tendência de **1,08 a 1,11**.

**O que significa:** a estação é estável, então um índice calculado com o histórico disponível no corte é confiável. A tendência é forte e constante: as médias do passado vão prever ~10% abaixo do real se nada for feito.

## 3. As três réguas

| Régua | Previsão do total dos 3 meses | Pergunta |
|---|---|---|
| MM12 | média mensal dos últimos 12 meses × 3 | O nível médio do ano basta? |
| MM12 × índice | média dos últimos 12 meses × soma dos índices dos 3 meses previstos | Ajustar pela estação melhora? |
| MM3 | média dos últimos 3 meses × 3 | O passado recente importa mais que o ano? |

`erro_relativo` compara com a MM12: abaixo de 1 = melhor que a MM12.

In [ ]:
GROUPS = {"produto": ["sk_product"], "categoria_loja": ["category_name", "sk_location"]}
forecasts_all = add_forecasts(panel_all, totals, GROUPS)
forecasts = forecasts_all[forecasts_all["cutoff"].isin(CUTOFFS)].copy()

REGUAS = {"MM12": "pred_mm12", "MM12 × índice": "pred_mm12_idx", "MM3": "pred_mm3"}
error_table(forecasts, REGUAS)

**Resultado**

| Régua | WAPE | Viés | Erro relativo à MM12 |
|---|---|---|---|
| MM12 | 99,2% | −8,1% | 1,00 |
| **MM12 × índice** | **93,1%** | −11,7% | **0,94** |
| MM3 | 120,5% | −1,9% | 1,21 |

**O que significa:**

- **O ajuste sazonal ajuda** (6% menos erro que a MM12 pura).
- **A MM3 é a pior:** numa série que vende em ~3 meses do ano, a média de 3 meses oscila entre zero e picos.
- **Por série, o erro fica perto de 100%**, o mesmo que prever sempre zero. Não é defeito das réguas: o ruído de venda de uma série é do tamanho da própria média (notebook 01). O erro por série sozinho não serve para comparar réguas; as seções 9 e 10 medem na soma.
- **Todas preveem abaixo do real**, efeito do crescimento (seção 6). A MM12 pura parece menos enviesada só porque os erros para cima em alguns cortes compensam os erros para baixo em outros (seção 5).

**Em palavras simples:** das três formas simples de prever (média do ano, média do ano ajustada pela época, média dos últimos 3 meses), a que usa só os últimos 3 meses é a pior — como tentar prever as vendas do ano todo olhando só para um trimestre parado, sem saber se caiu numa época fraca ou forte. Ajustar pela época do ano já ajuda um pouco. Mas nenhuma das três é boa o bastante sozinha: falta corrigir o crescimento (seção 6) e usar o grupo em vez do produto isolado (seção 7).

## 4. Erro por frequência de venda

A frequência é a divisão que mais diferencia as séries (notebook 02). A expectativa era a MM3 ir mal nas séries que vendem em poucos meses do ano.

In [ ]:
by_frequency = error_table(forecasts, REGUAS, by=["frequency_class"])
display(by_frequency.pivot(index="frequency_class", columns="regua", values="wape")[list(REGUAS)])
by_frequency.pivot(index="frequency_class", columns="regua", values="vies")[list(REGUAS)]

**Resultado** — WAPE e viés da MM12 × índice

| Frequência no corte | WAPE | Viés |
|---|---|---|
| Alta (5+ meses com venda) | 107,9% | **+50,5%** |
| Média (3–4) | 89,7% | −4,6% |
| Baixa (1–2) | 88,9% | **−53,2%** |

A MM3 é a pior nas três frequências.

**O que significa:** as séries que venderam em muitos meses no último ano vendem **menos** depois (a régua prevê 50% a mais), e as que venderam pouco vendem **mais** (a régua prevê 53% a menos). É o que acontece quando a diferença entre séries é acaso: quem teve um ano bom volta para a média, e quem teve um ano ruim também. A frequência medida no passado não descreve a série, descreve a sorte do último ano.

**Em palavras simples:** é o mesmo efeito de um jogador de basquete que acerta 9 em 10 arremessos numa partida — na próxima, ele tende a voltar ao seu normal, não a manter os 90%. Aqui, os produtos que "tiveram sorte" e venderam em muitos meses no ano passado tendem a vender menos no período seguinte, e os que "tiveram azar" tendem a vender mais. Não é queda nem recuperação real do produto: é só a sorte se equilibrando. Confiar demais no quanto um produto vendeu recentemente engana a previsão.

## 5. Erro por mês de corte e por categoria

O erro muda conforme a época do ano prevista? Alguma categoria foge do padrão?

In [ ]:
by_cutoff = error_table(forecasts, REGUAS, by=["cutoff"])
display(by_cutoff.pivot(index="cutoff", columns="regua", values="wape")[list(REGUAS)])
by_cutoff.pivot(index="cutoff", columns="regua", values="vies")[list(REGUAS)]

In [ ]:
by_category = error_table(forecasts, REGUAS, by=["category_name"])
by_category.pivot(index="category_name", columns="regua", values="wape")[list(REGUAS)].sort_values("MM12 × índice")

**Resultado**

- **Por corte:** a MM12 pura vai de +107% de viés (corte em maio, prevendo junho a agosto, baixa temporada) a −39% (corte em dezembro, prevendo janeiro a março, alta temporada). Com o índice, o viés fica entre −8% e −16% em todos os cortes.
- O WAPE é maior nos cortes que preveem a baixa temporada (~120% em abril a junho) do que nos que preveem a alta (~80% em outubro a dezembro): com menos vendas, sobram mais alvos zerados.
- **Por categoria:** todas ficam entre 91% e 94% com a MM12 × índice, exceto Pesca (72%, só 6 séries).

**O que significa:** sem o índice, a MM12 erra sistematicamente para cima na baixa temporada e para baixo na alta. O índice resolve isso. Nenhuma categoria foge do padrão.

## 6. Viés e tendência

Com a demanda crescendo ~15% ao ano, médias do passado tendem a prever abaixo do real. **Teste:** multiplicar a MM12 × índice pelo fator de tendência do corte (seção 2) remove o viés e reduz o erro?

Se sim, a tendência precisa entrar como feature: o modelo não tem como saber que a demanda está crescendo só olhando médias.

In [ ]:
TENDENCIA = {"MM12 × índice": "pred_mm12_idx", "MM12 × índice × tendência": "pred_mm12_idx_trend"}

display(error_table(forecasts, TENDENCIA, reference="MM12 × índice"))
trend_by_cut = error_table(forecasts, TENDENCIA, by=["cutoff"], reference="MM12 × índice")
trend_by_cut.pivot(index="cutoff", columns="regua", values="vies")[list(TENDENCIA)]

**Resultado**

| Régua | WAPE por série | Viés |
|---|---|---|
| MM12 × índice | 93,1% | −11,7% |
| MM12 × índice × tendência | 95,5% | **−2,8%** |

Com a tendência, o viés fica entre −7% e +1% em todos os cortes, contra −8% a −16% sem ela.

**O que significa:** **a correção de tendência elimina quase todo o viés.** O WAPE por série piora um pouco, mas isso é efeito da métrica (explicado na seção 10): na soma das séries, a tendência reduz muito o erro (seção 9). **A tendência precisa ser feature**, porque médias do passado não têm como saber que a demanda está crescendo.

**Em palavras simples:** sem corrigir o crescimento, a previsão fica sistematicamente "atrasada" — como prever o preço da gasolina de hoje olhando só para o preço de alguns meses atrás, sem levar em conta que ele só sobe. Adicionar essa correção quase zera esse atraso (o viés cai de −11,7% para −2,8%). O erro por produto (WAPE) piora um pouquinho de propósito, porque essa medida também dá uma vantagem escondida para quem chuta baixo (seção 10 explica isso); o que importa é que, na soma, a correção reduz bastante o erro (seção 9).

## 7. De onde vem o nível: da série, do produto ou da categoria?

As réguas acima usam a média **da própria série**. Como as séries são muito parecidas entre si, vale testar se a média de um grupo maior prevê melhor. As três opções usam o mesmo índice sazonal; só muda de onde vem a média mensal:

| Nível | Média mensal usada |
|---|---|
| Série | MM12 do próprio produto naquele local |
| Produto nos 6 locais | MM12 do mesmo produto, na média dos locais |
| Categoria no local | MM12 média dos produtos da categoria naquele local — a mesma previsão para todos eles |

**Como ler:** se o grupo empatar ou vencer a série, a diferença entre séries é quase toda acaso. Nesse caso, as features da própria série valem pouco, e o foco da Etapa 3 vai para o comportamento do grupo.

In [ ]:
NIVEIS = {
    "série": "pred_mm12_idx",
    "produto nos 6 locais": "pred_produto_idx",
    "categoria no local": "pred_categoria_loja_idx",
}

display(error_table(forecasts, NIVEIS, reference="série"))
by_frequency_level = error_table(forecasts, NIVEIS, by=["frequency_class"], reference="série")
by_frequency_level.pivot(index="frequency_class", columns="regua", values="wape")[list(NIVEIS)]

**Resultado**

| Origem do nível (× índice) | WAPE | Erro relativo à série |
|---|---|---|
| Série | 93,1% | 1,00 |
| Produto nos 6 locais | 86,8% | 0,93 |
| **Categoria no local** | **85,6%** | **0,92** |

Na frequência alta, a diferença é maior: a série erra 107,9% e a categoria no local, 86,1%. O viés é o mesmo nas três, porque os totais são iguais.

**O que significa:** **prever pela média do grupo erra menos do que usar o histórico da própria série**, e prever pela categoria é tão bom quanto pelo mesmo produto nos outros locais. A série sozinha adiciona mais ruído que informação.

**Em palavras simples:** apostar no comportamento do grupo (todos os produtos da categoria, naquela loja) funciona melhor do que tentar adivinhar o comportamento de cada produto sozinho — mesmo sabendo o que aquele produto específico vendeu no passado. É como prever a nota de um aluno num teste: a média da turma costuma prever melhor a nota individual do que o histórico de provas antigas daquele aluno específico, quando as notas de cada um variam muito de prova para prova.

## 8. Quem vendeu mais continua vendendo mais?

Outra forma de ver a mesma pergunta, de dois jeitos.

**Primeiro:** em cada corte, as séries são divididas em 5 grupos iguais pela venda dos 12 meses anteriores (1 = as 20% que menos venderam; 5 = as 20% que mais venderam). Se produtos que vendem mais continuassem vendendo mais, a venda dos 3 meses seguintes subiria do grupo 1 para o grupo 5.

**Segundo:** compara o volume dos últimos 12 meses com o dos 12 anteriores, em três níveis:

- **correlação:** 1 = quem vendeu mais num ano vende mais no outro; 0 = sem relação;
- **permanência no topo:** das 20% unidades que mais venderam no primeiro ano, quantas continuam nas 20% do segundo. Ao acaso, seriam 20%.

In [ ]:
display(past_volume_quintiles(forecasts))

persistence = pd.DataFrame(
    [
        {"unidade": "série (produto × local)", **year_over_year_persistence(complete, LAST_MONTH, ["sk_product", "sk_location"], DATE_COL)},
        {"unidade": "produto (6 locais somados)", **year_over_year_persistence(complete, LAST_MONTH, ["sk_product"], DATE_COL)},
        {
            "unidade": "categoria × local",
            **year_over_year_persistence(complete.join(products, "sk_product"), LAST_MONTH, ["category_name", "sk_location"], DATE_COL),
        },
    ]
).set_index("unidade")

persistence

**Resultado**

| Grupo pela venda dos 12 meses anteriores | Venda média nos 12 meses anteriores | Venda média nos 3 meses seguintes |
|---|---|---|
| 1 · as 20% que menos venderam | 5,9 | **5,8** |
| 2 | 13,8 | **5,7** |
| 3 | 19,9 | **5,8** |
| 4 | 26,7 | **5,6** |
| 5 · as 20% que mais venderam | 39,5 | **5,8** |

Quem vendeu quase 7× mais no ano anterior vende **exatamente o mesmo** nos 3 meses seguintes.

| Unidade | Correlação entre anos | Continuam no top 20% | Ao acaso |
|---|---|---|---|
| Série (produto × local) | −0,003 | 19,4% | 20% |
| Produto (6 locais somados) | −0,027 | 20,8% | 20% |
| Categoria × local | 0,996 | 92,3% | 20% |

**O que significa:** **nenhuma série e nenhum produto vende consistentemente mais que outro.** Quem ficou entre os 20% que mais venderam num ano tem a mesma chance de repetir que qualquer outro. A categoria × local se repete só porque as categorias têm quantidades diferentes de produtos: por série, todas vendem ~22 a 25 unidades por ano (notebook 02). Na prática, **todas as séries têm a mesma demanda esperada**, e as diferenças observadas são acaso. Isso explica as seções 4 e 7.

**Em palavras simples:** essa é a prova mais direta de que "quem vendeu mais no passado" não é quem "vai vender mais no futuro". Os 20% de produtos que mais venderam num ano venderam quase 7 vezes mais que os 20% que menos venderam — mas no período seguinte, todos os grupos venderam praticamente o mesmo (~5,7 a 5,8 unidades). É como separar pessoas pelo resultado de uma jogada de dados: quem tirou os números mais altos numa rodada não tem mais chance de tirar números altos na próxima rodada. A tabela de correlação confirma isso: por produto, ela é praticamente zero (nenhuma relação entre o que vendeu num ano e no outro); só aparece uma relação forte quando somamos por categoria × local, e isso é só porque algumas categorias têm mais produtos que outras, não porque um produto específico "puxa" a venda.

## 9. Erro quando as séries são somadas

A previsão de cada série erra muito por natureza: o ruído de venda é do tamanho da própria média (notebook 01). Mas quem usa a previsão normalmente soma séries, como na compra de uma categoria para uma loja ou no planejamento de um local. Os erros se compensam em parte quando as séries são somadas.

Cada régua é somada no nível indicado, por corte, antes de medir o erro.

In [ ]:
TODAS = {
    **REGUAS,
    "MM12 × índice × tendência": "pred_mm12_idx_trend",
    "categoria no local × índice": "pred_categoria_loja_idx",
    "categoria no local × índice × tendência": "pred_categoria_loja_idx_trend",
}

levels = {
    "série (produto × local)": ["sk_product", "sk_location"],
    "produto (6 locais)": ["sk_product"],
    "categoria × local": ["category_name", "sk_location"],
    "local": ["sk_location"],
    "total": [],
}
aggregated = pd.concat(
    [aggregated_error_table(forecasts, TODAS, level).assign(nivel=name) for name, level in levels.items()]
)
aggregated.pivot(index="regua", columns="nivel", values="wape")[list(levels)].loc[list(TODAS)]

**Resultado** — WAPE depois de somar as séries

| Régua | Série | Produto (6 locais) | Categoria × local | Local | Total |
|---|---|---|---|---|---|
| MM12 | 99,2% | 49,8% | 36,3% | 35,5% | 35,3% |
| MM12 × índice | 93,1% | 39,0% | 14,5% | 12,0% | 11,7% |
| MM3 | 120,5% | 65,8% | 49,5% | 48,7% | 48,5% |
| MM12 × índice × tendência | 95,5% | 39,1% | **10,7%** | **6,1%** | **3,3%** |
| Categoria no local × índice | 85,6% | 35,3% | 14,5% | 12,0% | 11,7% |
| Categoria no local × índice × tendência | 86,6% | **34,8%** | **10,7%** | **6,1%** | **3,3%** |

**O que significa:**

- **Somando, os erros de cada série se compensam:** de ~90% por série para ~11% por categoria × local e ~6% por local. A previsão serve bem para compra por categoria e planejamento por local.
- **Na soma, o que sobra é o viés**, e a tendência é o que mais reduz o erro: de 12,0% para 6,1% por local e de 11,7% para 3,3% no total.
- As réguas de série e de categoria empatam a partir de categoria × local, porque a soma é a mesma. A diferença entre elas aparece só por série e por produto.

**Em palavras simples:** pense numa pesquisa eleitoral: perguntar a 1 pessoa só dá um retrato muito impreciso; perguntar a milhares de pessoas dá um retrato bem mais confiável, porque os erros individuais de cada resposta se cancelam. É o mesmo aqui — errar bastante ao prever a venda de *um* produto (~90%) é esperado, mas ao somar dezenas de produtos numa categoria, ou centenas numa loja, os exageros para cima e para baixo se cancelam, e sobra um erro pequeno (11% e 6%). Isso é ótimo porque, na prática, ninguém decide estoque produto a produto — decide por categoria ou por loja, que é justamente onde a previsão é mais confiável.

## 10. A régua final

Junta o que as seções 6 a 9 mostraram: **média do grupo** (a série sozinha não informa o nível), **índice sazonal** e **fator de tendência**. A tabela compara as réguas nos dois jeitos de medir que importam: o erro por série e o erro somado por categoria × local e por local, onde a previsão é usada.

**Por que olhar os dois:** numa série com metade dos alvos em zero, o WAPE por série premia previsões abaixo da média. Prever um pouco menos acerta mais zeros, mesmo errando o total. Por isso uma régua sem viés pode ter WAPE por série maior e ainda assim ser a melhor na soma. O viés e o erro somado mostram quem acerta o total.

In [ ]:
FINAIS = {
    "MM12 × índice": "pred_mm12_idx",
    "MM12 × índice × tendência": "pred_mm12_idx_trend",
    "categoria no local × índice": "pred_categoria_loja_idx",
    "categoria no local × índice × tendência": "pred_categoria_loja_idx_trend",
}

final = error_table(forecasts, FINAIS, reference=None).set_index("regua")[["wape", "vies"]].rename(
    columns={"wape": "wape_serie", "vies": "vies"}
)
for name, level in {"categoria × local": ["category_name", "sk_location"], "local": ["sk_location"]}.items():
    final[f"wape_{name}"] = aggregated_error_table(forecasts, FINAIS, level).set_index("regua")["wape"]

final

**Resultado**

| Régua | WAPE por série | Viés | WAPE categoria × local | WAPE local |
|---|---|---|---|---|
| MM12 × índice | 93,1% | −11,7% | 14,5% | 12,0% |
| MM12 × índice × tendência | 95,5% | −2,8% | 10,7% | 6,1% |
| Categoria no local × índice | 85,6% | −11,7% | 14,5% | 12,0% |
| **Categoria no local × índice × tendência** | **86,6%** | **−2,8%** | **10,7%** | **6,1%** |

**O que significa:** a **categoria no local × índice × tendência** é a melhor régua: quase o menor erro por série, sem viés e o menor erro na soma. O WAPE por série de 85,6% (sem tendência) é um pouco menor só porque prever abaixo acerta mais meses zerados, e isso custa 4 pontos por categoria × local e 6 por local.

**Em palavras simples:** por que a régua com o erro por série levemente maior (86,6% em vez de 85,6%) é considerada a melhor? Porque o erro por série tem uma "pegadinha": ele dá vantagem escondida para quem sempre chuta um pouco abaixo do real, já que muitos produtos vendem zero, e chutar baixo acerta mais desses zeros. Uma régua pode "parecer" boa nessa conta só por prever sempre pouco, mesmo errando o total real. Olhando o viés (que mostra se a régua chuta muito pra baixo ou pra cima) e o erro depois de somar os produtos (onde é usada de verdade, por categoria e por loja), a régua com o ajuste de tendência é claramente melhor nas duas.

## 11. A faixa de venda de cada produto

A previsão por produto entrega três números: a **quantidade esperada** (a régua final) e dois **limites de venda**. Os limites existem porque a venda de um produto sozinho varia muito em torno do esperado.

**Como a faixa é calculada, sem olhar o futuro:**

1. Em 24 cortes **anteriores** ao período de teste, as previsões são divididas em 10 grupos por tamanho (das menores às maiores).
2. Em cada grupo, olha-se quanto de fato se vendeu e guardam-se os valores que 50%, 75% e 90% das vendas não ultrapassaram (`q_50`, `q_75`, `q_90`).
3. Nos 12 cortes de teste, cada previsão recebe os valores do grupo do seu tamanho.

| Número entregue | Coluna | Leitura |
|---|---|---|
| Quantidade esperada | `pred_categoria_loja_idx_trend` | Venda média esperada nos 3 meses |
| Limite de 3 em cada 4 vezes | `q_75` | A venda fica até esse valor em 3 de cada 4 vezes |
| Limite de 9 em cada 10 vezes | `q_90` | A venda fica até esse valor em 9 de cada 10 vezes |

**Por que limites, e não uma faixa "de X a Y":** metade dos alvos é zero, então o limite de baixo de qualquer faixa é sempre 0. O que informa é até onde a venda pode ir.

A calibração termina 3 meses antes do primeiro corte de teste, então todas as vendas usadas já tinham acontecido.

In [ ]:
FINAL = "pred_categoria_loja_idx_trend"

calibration = forecasts_all[forecasts_all["cutoff"].isin(CALIBRATION_CUTOFFS)]
buckets = fit_interval_buckets(calibration, FINAL)
forecasts = apply_interval_buckets(forecasts, buckets, FINAL)

buckets.round(2)

**Resultado** — o que se vendeu de fato, por tamanho da quantidade esperada (24 cortes de calibração)

| Quantidade esperada | Até q_50 | Até q_75 (3 em 4) | Até q_90 (9 em 10) |
|---|---|---|---|
| 0 a 2,1 | 0 | 3 | 8 |
| 3,1 a 3,7 | 0 | 6 | 10 |
| 4,5 a 5,1 | 3 | 9 | 14 |
| 6,7 a 15 | 6 | 11 | 17 |

**O que significa:** com quantidade esperada de até ~3,7 unidades, mais da metade das vezes o produto não vende nada nos 3 meses (q_50 = 0). Mesmo nas previsões maiores, a venda pode chegar a 17 unidades em 1 de cada 10 vezes. Os limites transformam essa variação em números para decidir estoque.

**Em palavras simples:** como ler essa tabela: pegue a linha da quantidade esperada mais parecida com a sua previsão — por exemplo, "4,5 a 5,1". Ela diz: metade das vezes esse produto vende 3 unidades ou menos (`q_50` = 3, incluindo muitos casos de venda zero dentro dessa metade), 3 em cada 4 vezes a venda não passa de 9 unidades, e 9 em cada 10 vezes não passa de 14. Repare que produtos "parecidos" no papel (mesma quantidade esperada) podem vender de 0 a 14 unidades — é essa variação natural que os limites ajudam a enxergar, em vez de esconder atrás de um único número médio.

## 12. Os limites acertam?

Se os limites estão bem calibrados, nos 12 cortes de teste a venda real deve ficar **até `q_50` em ~50%**, **até `q_75` em ~75%** e **até `q_90` em ~90%** das vezes. Um pouco acima é esperado em `q_50`, porque a venda é um número inteiro e muitos valores empatam em zero.

In [ ]:
display(interval_coverage(forecasts))
display(interval_coverage(forecasts, by=["frequency_class"]))
interval_coverage(forecasts, by=["cutoff"])

**Resultado**

| Limite | Prometido | Nos 12 cortes de teste |
|---|---|---|
| q_50 | ~50% | 51,8% |
| q_75 (3 em cada 4) | ~75% | **75,2%** |
| q_90 (9 em cada 10) | ~90% | **89,6%** |

- Igual nas três frequências de venda: alta 89,4%, média 89,8% e baixa 89,7% até q_90.
- Por corte, até q_90 vai de **85,9%** (corte em dezembro, prevendo janeiro a março) a **91,7%**. Nos cortes que preveem a alta temporada, a venda varia um pouco mais do que os limites preveem.

**O que significa:** **os limites estão bem calibrados** e podem ser entregues com confiança. Um refinamento possível é calibrar os limites separadamente para a alta temporada.

**Em palavras simples:** isso é o "teste de honestidade" dos tetos calculados na seção 11. Prometemos que a venda real ficaria dentro do teto menor em 75% das vezes e dentro do teto maior em 90% das vezes — e ao conferir com o que de fato aconteceu nos 12 meses de teste (dados que a régua nunca tinha visto), os números bateram quase exatamente (75,2% e 89,6%). Ou seja, pode confiar nesses tetos para decidir quanto manter em estoque, sabendo que eles vão errar para menos só raramente.

## 13. Como fica a entrega por produto

Um recorte real: motores da Loja Leão, previsão feita em novembro/2025 para dezembro a fevereiro, com o que de fato foi vendido. Cada linha é um **SKU** (uma variante do produto): o mesmo nome pode aparecer em SKUs diferentes.

In [ ]:
EXAMPLE_CUTOFF = add_months(CUTOFFS[0], 5)

delivery = forecasts[
    (forecasts["cutoff"] == EXAMPLE_CUTOFF)
    & (forecasts["location_name"] == "Loja Leão")
    & (forecasts["category_name"] == "Motores")
].sort_values("sku")

# O motor usado como exemplo no guia (SKU LHN-616875), seguido de outros motores da mesma loja.
# O nome de produto se repete entre variantes; o SKU identifica cada série.
example = delivery["sku"] == "LHN-616875"
sample = pd.concat([delivery[example], delivery[~example].head(7)])

display(
    sample[["sku", "product_name", "location_name", FINAL, "q_75", "q_90", "target_3m"]]
    .rename(
        columns={
            "product_name": "produto",
            "location_name": "loja",
            FINAL: "quantidade_esperada",
            "q_75": "limite_3_em_4",
            "q_90": "limite_9_em_10",
            "target_3m": "vendido",
        }
    )
    .round(1)
)

# Nos 112 motores da loja nesse corte, quantos ficaram dentro de cada limite.
interval_coverage(delivery)

**Resultado**

- O motor do guia (SKU **LHN-616875**) tinha quantidade esperada de **8,4**, limite de 3 em 4 de **11** e limite de 9 em 10 de **17**. Vendeu **16**: acima do limite de 3 em 4, dentro do de 9 em 10.
- Todos os 112 motores da loja recebem os mesmos números, e a venda real vai de 0 a 39.
- Nesse corte, os 112 motores ficaram até q_75 em 72% e até q_90 em 87% das vezes, um pouco abaixo do prometido: é um corte que prevê a alta temporada (seção 12).

**Observação sobre os dados:** o nome do produto se repete entre SKUs diferentes (há três "Motor de Popa 6225"), e alguns nomes não parecem de produto ("Genérico", "João da Silva"). As séries são identificadas pelo SKU, então isso não afeta a previsão, mas vale corrigir na origem para a entrega ficar legível.

## 14. A base de comparação

O quadro abaixo é a **referência para comparar qualquer previsão nova**, como o modelo da Etapa 4. Todas as linhas usam as mesmas 72.648 previsões (6.054 séries × 12 cortes).

| Métrica | O que mede | Melhor |
|---|---|---|
| `erro_produto` | Erro (WAPE) por produto × local | Menor, mas lembre da armadilha: premia prever abaixo |
| `erro_categoria_local`, `erro_local` | Erro depois de somar por categoria no local e por local | Menor |
| `vies` | Se erra sempre para o mesmo lado | Perto de zero |
| `erro_medio_unidades` | Erro médio por produto, em unidades | Menor |
| `desvio_poisson` | Erro por produto que não premia prever abaixo | Menor |

E, para os limites, a cobertura da seção 12: ~75% até `q_75` e ~90% até `q_90`.

**Em palavras simples, por que tantas métricas diferentes:** nenhuma métrica sozinha conta a história toda, então usamos um conjunto:
- **Erro por produto** é fácil de entender (de cada 100 unidades, quantas errou), mas tem uma armadilha: como quase metade dos produtos vende zero, uma régua que sempre "chuta baixo" acaba acertando mais desses zeros e parecendo melhor do que realmente é.
- **Erro depois de somar** (por categoria numa loja, ou por loja inteira) mostra o erro de verdade no jeito que a previsão é usada na prática — comprar estoque por categoria, planejar por loja.
- **Viés** mostra só uma coisa: a régua costuma prever a mais ou a menos, no total?
- **Erro médio em unidades** traduz o erro para um número concreto ("erra X unidades por produto"), mais fácil de visualizar que uma porcentagem.
- **Desvio de Poisson** é um jeito alternativo de medir o erro por produto que não cai na armadilha do "chutar baixo" — serve como conferência do erro por produto.

A régua só é considerada boa se for boa em **todas** essas visões ao mesmo tempo, não só na mais fácil de manipular.

In [ ]:
BASE = {
    "MM3": "pred_mm3",
    "MM12": "pred_mm12",
    "MM12 × índice": "pred_mm12_idx",
    "Régua final: categoria no local × índice × tendência": FINAL,
}

comparison_scorecard(forecasts, BASE)

### Como comparar a sua previsão com a base

1. Gere a sua previsão com uma linha por `cutoff`, `sk_product` e `sk_location`, para os mesmos 12 cortes de teste (`CUTOFFS`), e o total previsto dos 3 meses seguintes numa coluna, por exemplo `pred_modelo`.
2. Junte ao painel e rode o mesmo quadro:

```python
comparacao = forecasts.merge(minha_previsao, on=["cutoff", "sk_product", "sk_location"], how="left")
comparison_scorecard(comparacao, {"régua": FINAL, "meu modelo": "pred_modelo"})
```

3. Se entregar limites, nomeie-os `q_50`, `q_75` e `q_90` e use `interval_coverage`.

**Regra para dizer que é melhor:** menor erro por categoria no local e por local, viés mais perto de zero e desvio de Poisson menor, na maioria dos 12 cortes, e não só na média.

**Resultado** — a base de comparação

| Previsão | Erro por produto | Erro categoria × local | Erro por local | Viés | Erro médio (unidades) | Desvio de Poisson |
|---|---|---|---|---|---|---|
| MM3 | 120,5% | 49,5% | 48,7% | −1,9% | 6,93 | 75,8 |
| MM12 | 99,2% | 36,3% | 35,5% | −8,1% | 5,71 | 14,1 |
| MM12 × índice | 93,1% | 14,5% | 12,0% | −11,7% | 5,36 | 13,2 |
| **Régua final** | **86,6%** | **10,7%** | **6,1%** | **−2,8%** | **4,98** | **7,3** |

**O que significa:** a régua final é a melhor em **todas** as métricas, inclusive nas que não premiam prever abaixo (viés e desvio de Poisson). Por produto, ela erra em média **5 unidades** num total de ~5,8. **Estes são os números a superar** com a engenharia de features e o modelo.

**Em palavras simples:** a régua final (categoria no local × índice × tendência) vence as outras três em todas as colunas, o que não é comum — normalmente uma régua é melhor numa métrica e pior em outra. Isso reforça que é uma base confiável para comparar. Na prática: por produto, ela erra cerca de 5 unidades, o que é bastante perto do total médio vendido (~5,8) — prever a venda de *um* produto continua sendo difícil por natureza. Mas quando somada numa loja inteira, o erro cai para 6%, que é o número que realmente importa para o planejamento.

## Conclusão

**Em uma frase, para quem não trabalha com dados:** para prever quanto um produto vai vender numa loja, funciona melhor olhar para o comportamento do grupo dele (categoria + loja), ajustado pela época do ano e pelo crescimento das vendas, do que tentar adivinhar pelo histórico daquele produto específico — porque o histórico individual de cada produto é, na prática, resultado de sorte, não de um padrão que se repete.

### A base de comparação

**Régua final: categoria no local × índice sazonal × tendência**, com limites de venda por produto. Qualquer previsão nova é comparada nos mesmos 12 cortes e com a mesma função (seção 14):

| Medida | Régua | Uma previsão melhor precisa |
|---|---|---|
| Erro categoria × local | 10,7% | ficar abaixo |
| Erro por local | 6,1% | ficar abaixo |
| Viés | −2,8% | ficar mais perto de zero |
| Desvio de Poisson por produto | 7,3 | ficar abaixo |
| Erro médio por produto | 4,98 unidades | ficar abaixo |
| Limites (q_75 / q_90) | 75,2% / 89,6% | manter ~75% / ~90% |

A melhora precisa aparecer na maioria dos 12 cortes, e não só na média.

**Como avaliar:** sempre pelo viés e pelo erro somado, além do erro por série. O WAPE por série sozinho premia prever abaixo do real.

### O que isso muda na engenharia de features

| Grupo de features | Veredito | Por quê |
|---|---|---|
| **Nível do grupo** (média por série da categoria no local; média geral por série) | **Essencial** | Prevê melhor que a própria série (seção 7) |
| **Índice sazonal** dos 3 meses previstos | **Essencial** | Remove o viés de época, de +107% a −39% (seção 5) |
| **Tendência** (crescimento de 12 meses no corte; fator de tendência) | **Essencial** | Remove o viés e corta o erro somado pela metade (seções 6 e 9) |
| Histórico da própria série (MM3, MM12, meses com venda, meses desde a última venda) | **Provavelmente inútil** | O volume de uma série não se repete de um ano para o outro (seção 8); só testar para confirmar |
| Produto nos outros locais | **Provavelmente inútil** | O produto também não se repete entre anos, e empata com a categoria (seções 7 e 8) |
| Atributos (categoria, local, preço) | Baixa prioridade | Todas as categorias e locais vendem o mesmo por série |

**Expectativa honesta:** esta régua já captura quase tudo o que os dados permitem prever. Um modelo pode ganhar ajustando estação e tendência com mais precisão por grupo, mas o ganho deve ser pequeno. Se não superar a régua, a régua é a previsão recomendada.